### Problem:
Running `batch_labeling.py` multiple times on Philadelphia kept resulting in:

📊 Label Distribution:
**oracle_label**
- Ambiguous        931
- Contradictory    316
- Answerable        31

### Problem 1.5:
**Improvement:** Changed Solve logic, because previously, for a landmark to be "found," it must satisfy a strict intersection:$$\text{Candidate} = (\text{Distance} \leq 1500m) \cap (\text{OSM\_Tags} \in \text{LANDMARK\_GROUPS})$$
So if a landmark in Philadelphia is 200m away (well within your 1500m horizon) but its OpenStreetMap data is missing the specific amenity or shop tags we defined in config.py, the result of that intersection is Zero.

**Why it's still a problem:** Even after the change, running `batch_labeling.py` on Philadelphia resulted in:

📊 Label Distribution:
**oracle_label**

- Ambiguous        931
- Contradictory    21
- Answerable       135

which is **impossible**.

In [ ]:
import pandas as pd

# 1. Load the results from the batch run
results_path = "../data/philadelphia/philadelphia_silver_standard.parquet"
df_results = pd.read_parquet(results_path)

# 2. Filter for only the Ambiguous rows
ambiguous_df = df_results[df_results['oracle_label'] == 'Ambiguous']

print(f"Total Ambiguous rows found: {len(ambiguous_df)}")

# 3. Display the first 10 instructions and their extracted nouns
# This will show us exactly what the NLP model "saw"
print(ambiguous_df[['instruction', 'extracted_noun', 'candidate_count']].head(10))

# 4. Grab a specific one to test in our diagnostic tool
sample_row = ambiguous_df.iloc[0]
test_instruction = sample_row['instruction']
start_node = sample_row['start_node']

print(f"\n🚀 Ready to test Sample ID: {sample_row['sample_id']}")

Total Ambiguous rows found: 931
                                          instruction extracted_noun  \
0   Meet to the west of you, at Ben & Jerry's ice ...           None   
1   Meet me at the cafe north of you on the north ...           None   
2   Meet me at the historic memorial on the south ...           None   
3   Go south and a bit east. You'll find me at the...           None   
4   I am at the American Eagle Outfitters which is...           None   
7   Move near the river to see me at the bench on ...           None   
9   Meet me at a post box east of you on the south...           None   
11  Meet me at the playground by the southeast cor...           None   
12  I'm at the fast food restaurant on the west si...           None   
13  Meet me at the bicycle parking on the south si...           None   

    candidate_count  
0                69  
1                47  
2                77  
3                25  
4                45  
7                25  
9                 2  

There it is! We just found the Root Cause of the 931 Ambiguous rows.

## 🕵️ The "None" Noun Diagnosis
Look at extracted_noun column: It is **100% None.**

Because the noun extraction is failing to identify the landmark name (e.g., it missed "Ben & Jerry's" or "American Eagle Outfitters"), your SymbolicSolver is falling back to a Categorical-only search.

### Example of a Possible Chain Reaction for Instruction 0:

- Instruction: "Meet to the west of you, at Ben & Jerry's..."

- Extraction: Returns category: FOOD, noun: None.

- Solver: Says, "I don't have a specific name, so find me every POI that matches the category FOOD within 1500m."

- The Result: It finds 69 different food places nearby.

- The Final Label: Since 69 > 1, the solver marks it as **Ambiguous.**

In [ ]:
import os
import sys

# Move up one level from the 'notebooks' folder to the project root
project_root = os.path.dirname(os.path.abspath("")) 
if project_root not in sys.path:
    sys.path.append(project_root)

# Now we can import our project modules
import config
from src.extraction_utils import CategoricalMatcher
import re

print("✅ Project modules linked successfully!")

✅ Project modules linked successfully!


In [5]:
from src.extraction_utils import CategoricalMatcher
matcher = CategoricalMatcher()
import re

def extract_rvs_target_fixed(text: str) -> tuple:
    text_clean = text.replace("’", "'").replace(" ,", ",")

    # 1. Anchor Search
    anchor_pattern = r"\b(at|me at|is at|to)\b\s+(.*)"
    match = re.search(anchor_pattern, text_clean, re.IGNORECASE)
    if not match: return "UNKNOWN", "UNKNOWN"
    
    span = match.group(2)

    # 2. Clipping (REMOVED 'at' and 'the' from here)
    stops = [
        r"\b(?:on|near|across|which|is|south|north|west|east|corner|end|middle)\b",
        r",", r"\."
    ]
    
    earliest_stop = len(span)
    for stop_pattern in stops:
        s_match = re.search(stop_pattern, span, re.IGNORECASE)
        if s_match and s_match.start() < earliest_stop:
            earliest_stop = s_match.start()
    
    noun = span[:earliest_stop].strip()

    # 3. Cleanup (Handle the leading/trailing noise here instead)
    noun = re.sub(r"^(the|a|an)\s+", "", noun, flags=re.IGNORECASE)
    
    category = matcher.get_category(noun)
    return category, noun.strip()

In [6]:
test_cases = [
    "Meet to the west of you, at Ben & Jerry's ice cream.",
    "I am at the American Eagle Outfitters which is south.",
    "Meet me at the cafe north of you"
]

for t in test_cases:
    cat, noun = extract_rvs_target_fixed(t)
    print(f"Input: {t}")
    print(f"Output -> Category: {cat} | Noun: {noun}\n")

Input: Meet to the west of you, at Ben & Jerry's ice cream.
Output -> Category: UNKNOWN | Noun: the

Input: I am at the American Eagle Outfitters which is south.
Output -> Category: CLOTHES | Noun: American Eagle Outfitters

Input: Meet me at the cafe north of you
Output -> Category: CAFE | Noun: cafe



In [7]:
def extract_rvs_target_v3(text: str) -> tuple:
    # 1. Clean
    text_clean = text.replace("’", "'").replace(" ,", ",")

    # 2. THE FIX: Find the LAST 'at' or 'to' before the landmark
    # This ignores "at the west" and finds "at Ben & Jerry's"
    potential_anchors = [m.start() for m in re.finditer(r"\b(at|to|me at|is at)\b", text_clean, re.IGNORECASE)]
    
    if not potential_anchors:
        return "UNKNOWN", "UNKNOWN"
    
    # We take the last anchor found in the sentence
    start_idx = potential_anchors[-1]
    # Move the pointer past the anchor word itself (e.g., skip 'at ')
    span = re.sub(r"^(at|to|me at|is at)\s+", "", text_clean[start_idx:], flags=re.IGNORECASE)

    # 3. Clipping (No 'at' or 'the' here!)
    stops = [
        r"\b(?:on|near|across|which|is|south|north|west|east|corner|end|middle)\b",
        r",", r"\."
    ]
    
    earliest_stop = len(span)
    for stop_pattern in stops:
        s_match = re.search(stop_pattern, span, re.IGNORECASE)
        if s_match and s_match.start() < earliest_stop:
            earliest_stop = s_match.start()
    
    noun = span[:earliest_stop].strip()

    # 4. Cleanup
    noun = re.sub(r"^(the|a|an)\s+", "", noun, flags=re.IGNORECASE)
    
    # 5. Resolve Category
    category = matcher.get_category(noun)
    return category, noun.strip()

In [8]:
test_cases = [
    "Meet to the west of you, at Ben & Jerry's ice cream.",
    "I am at the American Eagle Outfitters which is south.",
    "Meet me at the cafe north of you"
]

for t in test_cases:
    cat, noun = extract_rvs_target_v3(t)
    print(f"Input: {t}")
    print(f"Result -> Category: {cat} | Noun: {noun}\n")

Input: Meet to the west of you, at Ben & Jerry's ice cream.
Result -> Category: SHOP | Noun: Ben & Jerry's ice cream

Input: I am at the American Eagle Outfitters which is south.
Result -> Category: CLOTHES | Noun: American Eagle Outfitters

Input: Meet me at the cafe north of you
Result -> Category: CAFE | Noun: cafe



Found desired output using `extract_rvs_target_v3`. Updating it in `extraction_utils.py`.
But tested it and again got an **impossible** result:
📊 Label Distribution:
**oracle_label**
- Ambiguous        666
- Contradictory    455
- Answerable       157

In [18]:
import pandas as pd
import os
import numpy as np
import pickle
import config
import re

# 1. Ensure project root and imports are accessible
# (Assumes you are in project_root/notebooks/)
import sys
project_root = os.path.dirname(os.path.abspath(""))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.oracle_engine import OracleEngine
from src.extraction_utils import extract_rvs_target # This will use the file version (v3)

# 2. Setup Distance Helper
def notebook_haversine(lat1, lon1, lat2, lon2):
    R = 6371000 
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1 
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    return R * (2 * np.arcsin(np.sqrt(a)))

# --- START THE DIAGNOSTIC ---
try:
    # 3. Load Graph and Oracle (The Missing Pieces)
    config.CURRENT_CITY = 'philadelphia'
    graph_path = os.path.join(project_root, config.get_graph_path())
    poi_path = os.path.join(project_root, config.get_poi_path())
    
    print(f"🔄 Loading Graph from {graph_path}...")
    with open(graph_path, 'rb') as f:
        G = pickle.load(f)
        
    print(f"🔄 Initializing Oracle from {poi_path}...")
    oracle = OracleEngine(G, poi_path)

    # 4. Load the Ambiguous results
    results_path = os.path.join(project_root, "data", "philadelphia", "philadelphia_silver_standard.parquet")
    df_results = pd.read_parquet(results_path)
    ambiguous_samples = df_results[df_results['oracle_label'] == 'Ambiguous'].head(15)
    
    print(f"\n🔬 Analyzing {len(ambiguous_samples)} Ambiguous samples...\n" + "="*60)

    for idx, row in ambiguous_samples.iterrows():
        instr = row['instruction']
        s_node = row['start_node']
        
        # A. Extraction
        cat, noun = extract_rvs_target(instr) 
        
        # B. Solver Simulation
        start_data = G.nodes[s_node]
        start_coords = (start_data['y'], start_data['x'])
        tags = config.LANDMARK_GROUPS.get(cat, {})
        
        # Use the INSTANCE 'oracle', not the CLASS 'OracleEngine'
        candidates = oracle.resolve_nearby_candidates(
            tags, start_coords[0], start_coords[1], 
            radius_m=1500,
            landmark_name=noun
        )
        
        print(f"ID {idx} | Instruction: {instr[:50]}...")
        print(f"  ↳ Extracted: ({cat}, '{noun}')")
        
        if len(candidates) > 1:
            dists = sorted([notebook_haversine(start_coords[0], start_coords[1], c['coords'][0], c['coords'][1]) for c in candidates])
            d1, d2 = dists[0], dists[1]
            
            print(f"  ↳ Found {len(candidates)} candidates.")
            print(f"  ↳ Distance Gap: Closest={d1:.1f}m | Second={d2:.1f}m")
            
            if d1 < 200 and d2 > 500:
                print("  ↳ 💡 INSIGHT: Nearest-Neighbor would resolve this to Answerable.")
        elif len(candidates) == 0:
            print("  ↳ ❌ FAILED: Zero candidates found.")
        else:
            print("  ↳ ✅ SUCCESS: Correctly resolved to 1 node.")
        print("-" * 60)

except Exception as e:
    print(f"❌ Execution Error: {e}")
    import traceback
    traceback.print_exc()

🔄 Loading Graph from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\philadelphia\philadelphia_graph.gpickle...


C:\Users\adan\AppData\Local\Temp\ipykernel_18424\158848522.py:35: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


🔄 Initializing Oracle from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\philadelphia\philadelphia_poi.pkl...
📍 Extracting coordinates from philadelphia 'centroid' column...

🔬 Analyzing 15 Ambiguous samples...
ID 0 | Instruction: Meet to the west of you, at Ben & Jerry's ice crea...
  ↳ Extracted: (SHOP, 'Ben & Jerry's ice cream')
  ↳ Found 69 candidates.
  ↳ Distance Gap: Closest=171.5m | Second=416.7m
------------------------------------------------------------
ID 1 | Instruction: Meet me at the cafe north of you on the north side...
  ↳ Extracted: (CAFE, 'cafe')
  ↳ Found 47 candidates.
  ↳ Distance Gap: Closest=333.0m | Second=359.2m
------------------------------------------------------------
ID 2 | Instruction: Meet me at the historic memorial on the south side...
  ↳ Extracted: (MONUMENT, 'historic memorial')
  ↳ Found 77 candidates.
  ↳ Distance Gap: Closest=341.6m | Second=412.5m
------------------------------------------------------------
ID 3 | Instr

Conclusion:
## 🧩 The "Philadelphia Problem": Why 666 Rows Were Ambiguous

The latest test in Philadelphia resulted in **666 Ambiguous** labels and **455 Contradictions**. By analyzing the data, we discovered that our AI was "too smart" for its own good; it was looking at every single landmark in the city, while the human who wrote the instructions only cared about the ones right in front of them, as stated in the RVS paper (RVS being the dataset we're using)

### The Proposed Fix: The "Salience Filter"
Implementing a **Salience Filter** to mimic human focus:

* **The "Cafe" Explosion:** If an instruction says "Meet at the cafe," and there are 47 cafes within 1.5km, the AI used to give up (Ambiguous). Now, it assumes the **nearest** cafe is the target, especially if it's within 200m.

* **The "North-ish" Problem:** Humans are bad at angles. If a user says "Go North" but the building is actually "North-West," our old code called it a "Contradiction." We now use a **45° Directional Wedge**, allowing for human error.

* **The "Chatty" Instructions:** When a user says "the recycling place and let's save the planet," we now use **Hard Boundary Clipping** to ignore the "fluff" and focus only on the word "recycling."

In [24]:
# 1. Force reload of all modules
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import pickle
import config
import src.utils as utils

# Ensure project root is in path
project_root = os.path.dirname(os.path.abspath(""))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import the updated classes/functions
from src.oracle_engine import OracleEngine
from src.symbolic_solver import SymbolicSolver
from src.extraction_utils import extract_rvs_target

# 2. Re-initialize with updated logic
config.CURRENT_CITY = 'philadelphia'
with open(os.path.join(project_root, config.get_graph_path()), 'rb') as f:
    G = pickle.load(f)

oracle = OracleEngine(G, os.path.join(project_root, config.get_poi_path()))
solver = SymbolicSolver(oracle)

# 3. Test the "Salience Filter" on known problematic IDs
# We will use ID 0 (Ben & Jerry's) and ID 11 (Playground)
test_ids = [0, 11]
df_phil = pd.read_parquet(os.path.join(project_root, "data/philadelphia/philadelphia_silver_standard.parquet"))

print(f"🚀 Testing Salience Filter & 45° Wedge Logic...\n" + "="*60)

for tid in test_ids:
    row = df_phil.iloc[tid]
    instr = row['instruction']
    start_node = row['start_node']
    
    # This now calls your updated solver.solve() which includes the Distance Ratio Test
    result = solver.solve(instr, start_node)
    
    print(f"ID {tid} | Instruction: {instr[:60]}...")
    print(f"  ↳ New State: {result['state']}")
    
    if result['state'] == 'Answerable':
        print(f"  ↳ ✅ SUCCESS: The Salience Filter resolved the ambiguity!")
    elif result['state'] == 'Ambiguous':
        print(f"  ↳ 🚩 STILL AMBIGUOUS: The landmarks were likely too close together.")
    else:
        print(f"  ↳ ℹ️ Result: {result['state']}")
    print("-" * 60)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


C:\Users\adan\AppData\Local\Temp\ipykernel_18424\2765541898.py:25: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


📍 Extracting coordinates from philadelphia 'centroid' column...
✅ Solver Initialized: Found 1 isolated graph components.
🚀 Testing Salience Filter & 45° Wedge Logic...
DEBUG: Solver extracted Noun: 'Ben & Jerry's ice cream'
ID 0 | Instruction: Meet to the west of you, at Ben & Jerry's ice cream on South...
  ↳ New State: Answerable
  ↳ ✅ SUCCESS: The Salience Filter resolved the ambiguity!
------------------------------------------------------------
DEBUG: Solver extracted Noun: 'playground by the southeast'
ID 11 | Instruction: Meet me at the playground by the southeast corner on Lanier ...
  ↳ New State: Answerable
  ↳ ✅ SUCCESS: The Salience Filter resolved the ambiguity!
------------------------------------------------------------


Success!
📊 Label Distribution:
**oracle_label**
- Answerable       1035
- Contradictory     161
- Ambiguous          82

Now optimizing (if needed):

In [25]:
# --- THE "TRUE ERROR" INVESTIGATOR ---
# Filter for what's left in the Ambiguous and Contradictory buckets
remaining_ambiguous = df_results[df_results['oracle_label'] == 'Ambiguous'].head(5)
remaining_contradictory = df_results[df_results['oracle_label'] == 'Contradictory'].head(5)

print(f"🕵️ Investigating 'Hard Failures' (True Errors)...\n")

print("--- 🚩 REMAINING AMBIGUOUS (The 'Twin' Problem) ---")
for idx, row in remaining_ambiguous.iterrows():
    res = solver.solve(row['instruction'], row['start_node'])
    print(f"ID {idx} | Noun: '{res['noun']}' | Candidates: {res['candidate_count']}")
    print(f"  ↳ Instruction: {row['instruction'][:80]}...")
    # These usually fail because d1 and d2 are too close (e.g., two Starbucks 50m apart)

print("\n--- ❌ REMAINING CONTRADICTORY (The 'Phantom' Problem) ---")
for idx, row in remaining_contradictory.iterrows():
    res = solver.solve(row['instruction'], row['start_node'])
    print(f"ID {idx} | Noun: '{res['noun']}' | Candidates: {res['candidate_count']}")
    if res['candidate_count'] == 0:
        print(f"  ↳ Reason: Landmark not found in OSM (Extraction or Data Gap).")
    else:
        print(f"  ↳ Reason: Directional mismatch or Reachability (Dead end).")
    print(f"  ↳ Instruction: {row['instruction'][:80]}...")

🕵️ Investigating 'Hard Failures' (True Errors)...

--- 🚩 REMAINING AMBIGUOUS (The 'Twin' Problem) ---
ID 0 | Noun: 'Ben & Jerry's ice cream' | Candidates: 69
  ↳ Instruction: Meet to the west of you, at Ben & Jerry's ice cream on South 40th Street, on the...
ID 1 | Noun: 'cafe' | Candidates: 47
  ↳ Instruction: Meet me at the cafe north of you on the north side of West Girard Avenue. BB&T b...
ID 2 | Noun: 'historic memorial' | Candidates: 77
  ↳ Instruction: Meet me at the historic memorial on the south side of Arch Street. It is a few s...
ID 3 | Noun: 'cafe' | Candidates: 25
  ↳ Instruction: Go south and a bit east. You'll find me at the cafe across the street from a ban...
ID 4 | Noun: 'American Eagle Outfitters' | Candidates: 45
  ↳ Instruction: I am at the American Eagle Outfitters which is in the middle of the block on Che...

--- ❌ REMAINING CONTRADICTORY (The 'Phantom' Problem) ---
ID 5 | Noun: 'this car sharing place here' | Candidates: 0
  ↳ Reason: Landmark not found in OSM